# AI Agents Multi-Agent Frameworks

## Why Multi-Agent Systems?

Single agents hit limits:
- Context window fills up on long tasks
- Hard to specialize one agent for everything
- No parallel execution

Multi-agent systems solve this via **division of labor** and **specialization**.

---

## Multi-Agent Design Patterns

### Supervisor Pattern
```
User → Supervisor → Researcher Agent
                 → Writer Agent
                 → Critic Agent
```

### Peer-to-Peer / Network Pattern
```
Agent A ↔ Agent B ↔ Agent C
```
Agents can call each other directly.

### Blackboard Pattern
All agents read/write to a shared state (blackboard). Useful for collaborative problem-solving.

---

## Framework Comparison

| Framework | Key Concept | Best For |
|-----------|-------------|----------|
| **CrewAI** | Role-playing crew | Structured workflows with defined roles |
| **AutoGen** | Conversable agents | Conversational multi-agent tasks |
| **Smolagents** | Code-first agents | Lightweight, simple agents |
| **PydanticAI** | Type-safe agents | Production apps needing type safety |
| **LangGraph** | Graph workflows | Complex stateful pipelines |

In [1]:
# ── CrewAI Example ────────────────────────────────────────────────────────────
# pip install crewai crewai-tools
import os

from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool

# Define Agents
researcher = Agent(
    role="Senior Research Analyst",
    goal="Research and synthesize accurate information on AI topics",
    backstory="Expert analyst with 10 years in AI research. You find the most relevant and recent information.",
    tools=[SerperDevTool()],
    llm="gpt-4o-mini",
    verbose=True
)

writer = Agent(
    role="Technical Writer",
    goal="Write clear, engaging technical content based on research",
    backstory="Expert technical writer who makes complex topics accessible.",
    llm="gpt-4o-mini",
    verbose=True
)

# Define Tasks
research_task = Task(
    description="Research the latest advancements in RAG (Retrieval Augmented Generation) in 2024-2025. Focus on key improvements.",
    expected_output="A comprehensive summary of RAG advancements with key techniques and papers.",
    agent=researcher
)

write_task = Task(
    description="Write a 300-word blog post about RAG advancements based on the research provided.",
    expected_output="An engaging blog post with clear sections and key takeaways.",
    agent=writer,
    context=[research_task]  # Uses output from research_task
)

# Create Crew
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,  # or Process.hierarchical
    verbose=True
)

# Run (comment out to avoid API costs in demo)
# result = crew.kickoff()
# print(result)
print("CrewAI setup complete uncomment crew.kickoff() to run")

/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/pydantic/plugin/_schema_validator.py:39: UserWarning: ImportError while loading the `logfire-plugin` Pydantic plugin, this plugin will not be installed.

ImportError("cannot import name 'ReadableLogRecord' from 'opentelemetry.sdk._logs' (/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/opentelemetry/sdk/_logs/__init__.py)")
  plugins = get_plugins()


CrewAI setup complete uncomment crew.kickoff() to run


In [2]:
# ── AutoGen Multi-Agent Conversation ─────────────────────────────────────────
# pip install pyautogen
import autogen

config_list = [{"model": "gpt-4o-mini", "api_key": os.getenv("OPENAI_API_KEY")}]

# Assistant Agent
assistant = autogen.AssistantAgent(
    name="AI_Assistant",
    llm_config={"config_list": config_list},
    system_message="You are a helpful AI assistant. Solve tasks step by step."
)

# User Proxy Agent (can execute code)
user_proxy = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER",  # "ALWAYS", "TERMINATE", "NEVER"
    max_consecutive_auto_reply=3,
    code_execution_config={
        "work_dir": "/tmp/autogen",
        "use_docker": False
    },
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "")
)

# Start conversation (comment out to avoid costs)
# user_proxy.initiate_chat(
#     assistant,
#     message="Write and execute a Python script that generates Fibonacci numbers up to 100"
# )

print("AutoGen setup complete")

In [3]:
# ── Group Chat with Multiple Agents ──────────────────────────────────────────
coder = autogen.AssistantAgent(
    name="Coder",
    llm_config={"config_list": config_list},
    system_message="You write clean, efficient Python code. Always include error handling."
)

reviewer = autogen.AssistantAgent(
    name="Reviewer",
    llm_config={"config_list": config_list},
    system_message="You review code for bugs, security issues, and best practices. Be critical."
)

group_chat = autogen.GroupChat(
    agents=[user_proxy, coder, reviewer],
    messages=[],
    max_round=6,
    speaker_selection_method="round_robin"  # or "auto"
)

manager = autogen.GroupChatManager(
    groupchat=group_chat,
    llm_config={"config_list": config_list}
)

print("Group chat with Coder + Reviewer agents ready")
# user_proxy.initiate_chat(manager, message="Create a class for a binary search tree")

In [4]:
# ── Smolagents: Minimal Agent ─────────────────────────────────────────────────
# pip install smolagents
from smolagents import CodeAgent, DuckDuckGoSearchTool, InferenceClientModel

model = InferenceClientModel(model_id="meta-llama/Llama-3.3-70B-Instruct")

agent = CodeAgent(
    tools=[DuckDuckGoSearchTool()],
    model=model,
    max_steps=5
)

# result = agent.run("What are the top 3 most downloaded Python packages this month?")
# print(result)
print("Smolagents CodeAgent ready")

Smolagents CodeAgent ready


In [5]:
# ── PydanticAI: Type-Safe Agent ───────────────────────────────────────────────
# pip install pydantic-ai
from pydantic_ai import Agent
from pydantic import BaseModel

class AnalysisResult(BaseModel):
    sentiment: str
    score: float
    key_topics: list[str]
    recommendation: str

analysis_agent = Agent(
    "openai:gpt-4o-mini",
    output_type=AnalysisResult,
    system_prompt="Analyze text and return structured insights."
)

# result = await analysis_agent.run("The new iPhone has amazing camera but terrible battery life")
# print(result.data)  # Guaranteed to be AnalysisResult
print("PydanticAI agent with typed output ready")

PydanticAI agent with typed output ready


/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/pydantic_ai/agent/__init__.py:415: PydanticAIDeprecationWarning: In v2.0, 'openai:' will resolve to the OpenAI Responses API by default. Use 'openai-chat:' to keep current Chat Completions behavior, or 'openai-responses:' to opt in early.
  self._model = models.infer_model(model)


## Additional Learning Resources

### Documentation
- [CrewAI Docs](https://docs.crewai.com/)
- [AutoGen Docs](https://microsoft.github.io/autogen/stable/)
- [Smolagents Docs](https://huggingface.co/docs/smolagents/)
- [PydanticAI Docs](https://ai.pydantic.dev/)

### Papers
- [Generative Agents (Park et al., 2023)](https://arxiv.org/abs/2304.03442)
- [AutoGen Paper (Wu et al., 2023)](https://arxiv.org/abs/2308.08155)
- [MetaGPT (Hong et al., 2023)](https://arxiv.org/abs/2308.00352)

### GitHub
- [CrewAI GitHub](https://github.com/joaomdmoura/crewAI)
- [AutoGen GitHub](https://github.com/microsoft/autogen)
- [Smolagents GitHub](https://github.com/huggingface/smolagents)